# 06 — Week 2 Projections

This notebook converts the updated weekly team-strength ratings into game level projections for the selected target week.

The original preseason notebooks and preseason projections remain frozen.

For each target week matchup, this notebook:

- loads the updated weekly team strengths from Notebook 04
- preserves the original preseason Week 2 projection as a baseline
- recalculates expected margin using weekly strength, historical home field advantage, and rest
- converts expected margin into win probability using historical NFL margin variability
- carries forward the frozen preseason scoring baseline created for Week 1
- adds the new in season offensive and defensive scoring evidence
- produces projected scores, model spreads, ATS comparisons, totals comparisons, and final output tables
- saves all outputs only inside the weekly layer

The market is used only for comparison. It never enters the model projection itself.


In [61]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import norm
from sklearn.linear_model import LinearRegression


## Paths and Weekly Settings

For Week 2, `TARGET_WEEK = 2`.

Future weeks use the same notebook by changing the target week after the preceding weekly notebooks have been rerun.


In [62]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"
WEEKLY_OUTPUT_DIR = PROJECT_ROOT / "weekly_projections" / "outputs"

SEASON = 2026
TARGET_WEEK = 2

WEEK_OUTPUT_DIR = (
    WEEKLY_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}"
)

WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

TARGET_SCHEDULE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_schedule.parquet"
)

PRESEASON_BASELINE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_preseason_baseline.parquet"
)

WEEK1_PROJECTIONS_PATH = (
    WEEKLY_DATA_DIR
    / "week_01_projections.parquet"
)


# Load Weekly Inputs

Notebook 04 supplies the updated ratings. Notebook 02 supplies the target week schedule and frozen preseason comparison.


In [63]:
weekly_strength = pd.read_parquet(WEEKLY_STRENGTH_PATH)
target_schedule = pd.read_parquet(TARGET_SCHEDULE_PATH)
preseason_baseline = pd.read_parquet(PRESEASON_BASELINE_PATH)

print("Weekly team strengths:", weekly_strength.shape)
print("Target-week games:", target_schedule.shape)
print("Frozen preseason games:", preseason_baseline.shape)

assert len(weekly_strength) == 32
assert weekly_strength["team"].nunique() == 32
assert len(target_schedule) == len(preseason_baseline)


Weekly team strengths: (32, 25)
Target-week games: (16, 46)
Frozen preseason games: (16, 16)


### Injury-adjusted game strength

For Week 2, the `home_weekly_team_strength` and `away_weekly_team_strength` values now include the conservative injury adjustment from Notebook 05.

The injury adjustment remains visible separately in the saved team-strength file, so we can distinguish:

**preseason → performance update → injury update → final Week 2 rating**


# Attach Updated Team Strength

The matchup calculation is identical in structure to the original preseason game prediction model. The only change is the source of team strength:

`preseason team_strength` → `weekly_team_strength`

This isolates the in-season update without changing the original model.


In [64]:
strength_lookup = weekly_strength[
    [
        "team",
        "injury_adjusted_team_strength",
        "pre_injury_weekly_team_strength",
        "preseason_team_strength",
        "team_strength_change",
        "weekly_offense_adjustment",
        "weekly_defense_adjustment"
    ]
].copy()

games = target_schedule.copy()

home_lookup = strength_lookup.rename(columns={
    "team": "home_team",
    "injury_adjusted_team_strength": "home_weekly_team_strength",
    "pre_injury_weekly_team_strength": "home_pre_injury_team_strength",
    "preseason_team_strength": "home_preseason_team_strength",
    "team_strength_change": "home_strength_change",
    "weekly_offense_adjustment": "home_weekly_offense_adjustment",
    "weekly_defense_adjustment": "home_weekly_defense_adjustment"
})

away_lookup = strength_lookup.rename(columns={
    "team": "away_team",
    "injury_adjusted_team_strength": "away_weekly_team_strength",
    "pre_injury_weekly_team_strength": "away_pre_injury_team_strength",
    "preseason_team_strength": "away_preseason_team_strength",
    "team_strength_change": "away_strength_change",
    "weekly_offense_adjustment": "away_weekly_offense_adjustment",
    "weekly_defense_adjustment": "away_weekly_defense_adjustment"
})

games = (
    games
    .merge(home_lookup, on="home_team", how="left", validate="many_to_one")
    .merge(away_lookup, on="away_team", how="left", validate="many_to_one")
)

games["weekly_neutral_strength_diff"] = (
    games["home_weekly_team_strength"]
    - games["away_weekly_team_strength"]
)

games["preseason_neutral_strength_diff"] = (
    games["home_preseason_team_strength"]
    - games["away_preseason_team_strength"]
)

assert games["home_weekly_team_strength"].notna().all()
assert games["away_weekly_team_strength"].notna().all()


# Home Field and Rest

These effects are recalculated using the same historical 2015–2025 methodology as the frozen preseason game prediction notebook.

They are not retuned using Week 1.


In [65]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & (pl.col("location") == "Home")
        & pl.col("result").is_not_null()
    )
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()
rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

print("Historical HFA:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest points/day:", round(REST_POINT_VALUE, 3))


Historical HFA: 1.764
Rest points/day: 0.181


In [66]:
games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"]
    - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"]
    * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["weekly_neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)


# Win Probabilities

Historical 2015–2025 game-margin variability is retained exactly as the uncertainty scale. Week 1 is not used to recalibrate probability confidence.


In [67]:
historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.50,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

print("Historical margin SD:", round(historical_margin_std, 3))


Historical margin SD: 14.201


# Preseason vs Weekly Margin Movement

The frozen preseason Week 2 projection is attached for diagnostics only.

This makes it possible to see exactly how much the Week 1 evidence changed each Week 2 matchup.


In [68]:
preseason_compare = preseason_baseline[
    [
        "game_id",
        "expected_home_margin",
        "predicted_winner",
        "predicted_win_probability"
    ]
].rename(columns={
    "expected_home_margin": "preseason_expected_home_margin",
    "predicted_winner": "preseason_predicted_winner",
    "predicted_win_probability": "preseason_predicted_win_probability"
})

games = games.merge(
    preseason_compare,
    on="game_id",
    how="left",
    validate="one_to_one"
)

games["margin_change_from_preseason"] = (
    games["expected_home_margin"]
    - games["preseason_expected_home_margin"]
)

display(
    games[
        [
            "away_team",
            "home_team",
            "preseason_expected_home_margin",
            "expected_home_margin",
            "margin_change_from_preseason",
            "preseason_predicted_winner",
            "predicted_winner",
            "predicted_win_probability"
        ]
    ].round(3)
)


,away_team,home_team,preseason_expected_home_margin,expected_home_margin,margin_change_from_preseason,preseason_predicted_winner,predicted_winner,predicted_win_probability
0,DET,BUF,2.561,2.912,0.350,BUF,BUF,0.581
1,CAR,ATL,4.965,5.896,0.930,ATL,ATL,0.661
2,CIN,HOU,5.313,4.455,-0.858,HOU,HOU,0.623
3,CLE,TB,6.515,8.087,1.572,TB,TB,0.715
4,GB,NYJ,-6.707,-4.202,2.504,GB,GB,0.616
5,IND,KC,2.353,5.643,3.290,KC,KC,0.654
6,JAX,DEN,3.439,-0.277,-3.716,DEN,JAX,0.508
7,LV,LAC,8.368,6.138,-2.230,LAC,LAC,0.667
8,MIA,SF,7.117,9.889,2.773,SF,SF,0.757
9,MIN,CHI,1.237,1.824,0.587,CHI,CHI,0.551


# Frozen Preseason Scoring Ratings

Week 1's saved projection artifact contains the preseason offensive/defensive scoring ratings and personnel scoring adjustments for all 32 teams.

Those values were created before the season and are reused here as the frozen scoring prior. We do not rebuild or modify the preseason notebooks.

The new weekly offense/defense signals are added separately afterward.


In [69]:
week1_frozen = pd.read_parquet(
    WEEK1_PROJECTIONS_PATH
)

home_scoring = week1_frozen[
    [
        "home_team",
        "home_projected_off_ppg",
        "home_projected_def_ppg",
        "home_off_personnel_z",
        "home_def_personnel_z"
    ]
].rename(columns={
    "home_team": "team",
    "home_projected_off_ppg": "preseason_projected_off_ppg",
    "home_projected_def_ppg": "preseason_projected_def_ppg",
    "home_off_personnel_z": "off_personnel_z",
    "home_def_personnel_z": "def_personnel_z"
})

away_scoring = week1_frozen[
    [
        "away_team",
        "away_projected_off_ppg",
        "away_projected_def_ppg",
        "away_off_personnel_z",
        "away_def_personnel_z"
    ]
].rename(columns={
    "away_team": "team",
    "away_projected_off_ppg": "preseason_projected_off_ppg",
    "away_projected_def_ppg": "preseason_projected_def_ppg",
    "away_off_personnel_z": "off_personnel_z",
    "away_def_personnel_z": "def_personnel_z"
})

scoring_prior = (
    pd.concat([home_scoring, away_scoring], ignore_index=True)
    .groupby("team", as_index=False)
    .agg(
        preseason_projected_off_ppg=("preseason_projected_off_ppg", "mean"),
        preseason_projected_def_ppg=("preseason_projected_def_ppg", "mean"),
        off_personnel_z=("off_personnel_z", "mean"),
        def_personnel_z=("def_personnel_z", "mean")
    )
)

assert len(scoring_prior) == 32
display(scoring_prior.head())


,team,preseason_projected_off_ppg,preseason_projected_def_ppg,off_personnel_z,def_personnel_z
0,ARI,22.191473,24.425844,-0.763765,-1.345326
1,ATL,22.032425,23.261103,0.507936,0.513715
2,BAL,25.007820,22.580763,0.442967,0.969729
3,BUF,25.942602,22.289034,1.060099,0.231615
4,CAR,20.533474,23.731353,-1.506345,-0.346889


# Weekly Scoring Update

The in-season offensive and defensive adjustments from Notebook 04 are already shrunk for sample size.

They are added to the frozen scoring prior:

- positive offensive adjustment raises expected scoring
- positive defensive adjustment represents points saved, so it lowers projected points allowed

This keeps the Week 1 scoring evidence separate and interpretable.


In [70]:
scoring_update = weekly_strength[
    [
        "team",
        "weekly_offense_adjustment",
        "weekly_defense_adjustment"
    ]
].merge(
    scoring_prior,
    on="team",
    how="left",
    validate="one_to_one"
)

scoring_update["weekly_projected_off_ppg"] = (
    scoring_update["preseason_projected_off_ppg"]
    + scoring_update["weekly_offense_adjustment"]
)

scoring_update["weekly_projected_def_ppg"] = (
    scoring_update["preseason_projected_def_ppg"]
    - scoring_update["weekly_defense_adjustment"]
)

home_scoring_update = scoring_update.rename(columns={
    "team": "home_team",
    "weekly_projected_off_ppg": "home_weekly_projected_off_ppg",
    "weekly_projected_def_ppg": "home_weekly_projected_def_ppg",
    "off_personnel_z": "home_off_personnel_z",
    "def_personnel_z": "home_def_personnel_z"
})

away_scoring_update = scoring_update.rename(columns={
    "team": "away_team",
    "weekly_projected_off_ppg": "away_weekly_projected_off_ppg",
    "weekly_projected_def_ppg": "away_weekly_projected_def_ppg",
    "off_personnel_z": "away_off_personnel_z",
    "def_personnel_z": "away_def_personnel_z"
})

games = (
    games
    .merge(
        home_scoring_update[
            [
                "home_team",
                "home_weekly_projected_off_ppg",
                "home_weekly_projected_def_ppg",
                "home_off_personnel_z",
                "home_def_personnel_z"
            ]
        ],
        on="home_team",
        how="left"
    )
    .merge(
        away_scoring_update[
            [
                "away_team",
                "away_weekly_projected_off_ppg",
                "away_weekly_projected_def_ppg",
                "away_off_personnel_z",
                "away_def_personnel_z"
            ]
        ],
        on="away_team",
        how="left"
    )
)


# Weekly Projected Total

The matchup total uses the same basic preseason scoring concept: each offense is paired with the opposing defense.

Because the weekly offense/defense inputs are already heavily shrunk, no additional Week 1 multiplier is applied here.

The final projected scores are forced to satisfy both:

- projected score difference = weekly expected margin
- projected score sum = weekly projected total


In [71]:
games["home_matchup_points"] = (
    games["home_weekly_projected_off_ppg"]
    + games["away_weekly_projected_def_ppg"]
) / 2

games["away_matchup_points"] = (
    games["away_weekly_projected_off_ppg"]
    + games["home_weekly_projected_def_ppg"]
) / 2

games["projected_total"] = (
    games["home_matchup_points"]
    + games["away_matchup_points"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2

display(
    games[
        [
            "away_team",
            "home_team",
            "projected_away_score",
            "projected_home_score",
            "projected_total",
            "expected_home_margin"
        ]
    ].round(2)
)


,away_team,home_team,projected_away_score,projected_home_score,projected_total,expected_home_margin
0,DET,BUF,23.54,26.45,49.99,2.91
1,CAR,ATL,20.07,25.96,46.03,5.90
2,CIN,HOU,21.64,26.09,47.73,4.45
3,CLE,TB,18.46,26.54,45.00,8.09
4,GB,NYJ,24.65,20.45,45.10,-4.20
5,IND,KC,20.25,25.90,46.15,5.64
6,JAX,DEN,22.69,22.41,45.10,-0.28
7,LV,LAC,18.40,24.54,42.94,6.14
8,MIA,SF,17.71,27.60,45.31,9.89
9,MIN,CHI,22.93,24.75,47.68,1.82


# Market Comparison

nflverse uses positive `spread_line` for a home favorite and negative for an away favorite. The values below are converted to standard sportsbook notation.

Market information is used only after the model projections have been completed.


In [72]:
games["home_spread"] = -games["spread_line"]
games["away_spread"] = games["spread_line"]

games["model_home_spread"] = -games["expected_home_margin"]
games["model_away_spread"] = games["expected_home_margin"]

games["home_ats_edge"] = (
    games["home_spread"]
    - games["model_home_spread"]
)

games["away_ats_edge"] = (
    games["away_spread"]
    - games["model_away_spread"]
)

games["ats_pick"] = np.where(
    games["home_ats_edge"] > games["away_ats_edge"],
    games["home_team"],
    np.where(
        games["away_ats_edge"] > games["home_ats_edge"],
        games["away_team"],
        "PUSH"
    )
)

games["ats_difference"] = (
    games[["home_ats_edge", "away_ats_edge"]]
    .max(axis=1)
)

games["total_difference_signed"] = (
    games["projected_total"]
    - games["total_line"]
)

games["total_difference"] = (
    games["total_difference_signed"].abs()
)

games["total_pick"] = np.where(
    games["total_difference_signed"] > 0,
    "OVER",
    np.where(
        games["total_difference_signed"] < 0,
        "UNDER",
        "PUSH"
    )
)


# Display Helpers


In [73]:
def format_spread(team, line):
    if pd.isna(line):
        return "N/A"
    if line > 0:
        return f"{team} +{line:.1f}"
    return f"{team} {line:.1f}"

def model_spread_display(row):
    if row["model_home_spread"] < 0:
        return format_spread(row["home_team"], row["model_home_spread"])
    if row["model_away_spread"] < 0:
        return format_spread(row["away_team"], row["model_away_spread"])
    return "PICK"

def market_spread_display(row):
    if pd.isna(row["spread_line"]):
        return "N/A"
    if row["home_spread"] < 0:
        return format_spread(row["home_team"], row["home_spread"])
    if row["away_spread"] < 0:
        return format_spread(row["away_team"], row["away_spread"])
    return "PICK"

def ats_pick_display(row):
    if row["ats_pick"] == "PUSH":
        return "PUSH"
    if row["ats_pick"] == row["home_team"]:
        return format_spread(row["home_team"], row["home_spread"])
    return format_spread(row["away_team"], row["away_spread"])

games["projected_score"] = (
    games["away_team"]
    + " "
    + games["projected_away_score"].round(1).astype(str)
    + " - "
    + games["home_team"]
    + " "
    + games["projected_home_score"].round(1).astype(str)
)

games["model_spread_display"] = games.apply(
    model_spread_display, axis=1
)

games["market_spread_display"] = games.apply(
    market_spread_display, axis=1
)

games["ats_pick_display"] = games.apply(
    ats_pick_display, axis=1
)


# Final Week Projection Board


In [74]:
projection_board = pd.DataFrame({
    "Matchup": games["away_team"] + " @ " + games["home_team"],
    "Projected Score": games["projected_score"],
    "SU Pick": games["predicted_winner"],
    "Win Prob": games["predicted_win_probability"],
    "Model Spread": games["model_spread_display"],
    "Market Spread": games["market_spread_display"],
    "ATS Pick": games["ats_pick_display"],
    "ATS Diff": games["ats_difference"],
    "Model Total": games["projected_total"],
    "Market Total": games["total_line"],
    "O/U Pick": games["total_pick"],
    "Total Diff": games["total_difference"],
    "Preseason Margin": games["preseason_expected_home_margin"],
    "Weekly Margin": games["expected_home_margin"],
    "Margin Change": games["margin_change_from_preseason"]
})

display(
    projection_board.style.format({
        "Win Prob": "{:.3f}",
        "ATS Diff": "{:.2f}",
        "Model Total": "{:.1f}",
        "Market Total": "{:.1f}",
        "Total Diff": "{:.1f}",
        "Preseason Margin": "{:.2f}",
        "Weekly Margin": "{:.2f}",
        "Margin Change": "{:+.2f}"
    }).hide(axis="index")
)


Matchup,Projected Score,SU Pick,Win Prob,Model Spread,Market Spread,ATS Pick,ATS Diff,Model Total,Market Total,O/U Pick,Total Diff,Preseason Margin,Weekly Margin,Margin Change
DET @ BUF,DET 23.5 - BUF 26.5,BUF,0.581,BUF -2.9,BUF -5.5,DET +5.5,2.59,50.0,54.5,UNDER,4.5,2.56,2.91,+0.35
CAR @ ATL,CAR 20.1 - ATL 26.0,ATL,0.661,ATL -5.9,CAR -2.5,ATL +2.5,8.40,46.0,43.5,OVER,2.5,4.97,5.90,+0.93
CIN @ HOU,CIN 21.6 - HOU 26.1,HOU,0.623,HOU -4.5,HOU -2.5,HOU -2.5,1.95,47.7,45.5,OVER,2.2,5.31,4.45,-0.86
CLE @ TB,CLE 18.5 - TB 26.5,TB,0.715,TB -8.1,TB -8.5,CLE +8.5,0.41,45.0,41.5,OVER,3.5,6.51,8.09,+1.57
GB @ NYJ,GB 24.7 - NYJ 20.4,GB,0.616,GB -4.2,GB -3.5,GB -3.5,0.70,45.1,44.5,OVER,0.6,-6.71,-4.20,+2.50
IND @ KC,IND 20.3 - KC 25.9,KC,0.654,KC -5.6,KC -6.5,IND +6.5,0.86,46.1,46.5,UNDER,0.4,2.35,5.64,+3.29
JAX @ DEN,JAX 22.7 - DEN 22.4,JAX,0.508,JAX -0.3,DEN -2.5,JAX +2.5,2.78,45.1,45.5,UNDER,0.4,3.44,-0.28,-3.72
LV @ LAC,LV 18.4 - LAC 24.5,LAC,0.667,LAC -6.1,LAC -6.5,LV +6.5,0.36,42.9,43.5,UNDER,0.6,8.37,6.14,-2.23
MIA @ SF,MIA 17.7 - SF 27.6,SF,0.757,SF -9.9,SF -12.5,MIA +12.5,2.61,45.3,44.5,OVER,0.8,7.12,9.89,+2.77
MIN @ CHI,MIN 22.9 - CHI 24.8,CHI,0.551,CHI -1.8,CHI -4.5,MIN +4.5,2.68,47.7,47.5,OVER,0.2,1.24,1.82,+0.59


# Biggest Model vs Market Differences


In [75]:
biggest_ats_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Spread",
            "Market Spread",
            "ATS Pick",
            "ATS Diff"
        ]
    ]
    .sort_values("ATS Diff", ascending=False)
    .head(5)
    .copy()
)

biggest_total_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Total",
            "Market Total",
            "O/U Pick",
            "Total Diff"
        ]
    ]
    .sort_values("Total Diff", ascending=False)
    .head(5)
    .copy()
)

print("BIGGEST ATS DIFFERENCES")
display(biggest_ats_differences.round(2))

print()
print("BIGGEST TOTAL DIFFERENCES")
display(biggest_total_differences.round(2))


BIGGEST ATS DIFFERENCES


,Matchup,Projected Score,Model Spread,Market Spread,ATS Pick,ATS Diff
1,CAR @ ATL,CAR 20.1 - ATL 26.0,ATL -5.9,CAR -2.5,ATL +2.5,8.40
6,JAX @ DEN,JAX 22.7 - DEN 22.4,JAX -0.3,DEN -2.5,JAX +2.5,2.78
9,MIN @ CHI,MIN 22.9 - CHI 24.8,CHI -1.8,CHI -4.5,MIN +4.5,2.68
8,MIA @ SF,MIA 17.7 - SF 27.6,SF -9.9,SF -12.5,MIA +12.5,2.61
0,DET @ BUF,DET 23.5 - BUF 26.5,BUF -2.9,BUF -5.5,DET +5.5,2.59



BIGGEST TOTAL DIFFERENCES


,Matchup,Projected Score,Model Total,Market Total,O/U Pick,Total Diff
11,PHI @ TEN,PHI 26.4 - TEN 17.8,44.14,39.5,OVER,4.64
0,DET @ BUF,DET 23.5 - BUF 26.5,49.99,54.5,UNDER,4.51
13,SEA @ ARI,SEA 25.3 - ARI 19.6,44.86,40.5,OVER,4.36
3,CLE @ TB,CLE 18.5 - TB 26.5,45.00,41.5,OVER,3.50
15,NYG @ LA,NYG 18.8 - LA 26.9,45.70,48.5,UNDER,2.80


In [76]:
biggest_ats_differences["ATS Diff"] = (
    biggest_ats_differences["ATS Diff"].round(2)
)

biggest_total_differences["Model Total"] = (
    biggest_total_differences["Model Total"].round(1)
)

biggest_total_differences["Market Total"] = (
    biggest_total_differences["Market Total"].round(1)
)

biggest_total_differences["Total Diff"] = (
    biggest_total_differences["Total Diff"].round(1)
)

# Biggest Changes from the Preseason Projection

This is one of the most important weekly diagnostics. It shows which games changed most because of new in season information.


In [77]:
biggest_projection_changes = (
    projection_board[
        [
            "Matchup",
            "Preseason Margin",
            "Weekly Margin",
            "Margin Change",
            "SU Pick",
            "Win Prob"
        ]
    ]
    .assign(
        Absolute_Change=lambda x: x["Margin Change"].abs()
    )
    .sort_values("Absolute_Change", ascending=False)
    .drop(columns="Absolute_Change")
)

display(biggest_projection_changes.round(3))


,Matchup,Preseason Margin,Weekly Margin,Margin Change,SU Pick,Win Prob
6,JAX @ DEN,3.439,-0.277,-3.716,JAX,0.508
5,IND @ KC,2.353,5.643,3.290,KC,0.654
8,MIA @ SF,7.117,9.889,2.773,SF,0.757
4,GB @ NYJ,-6.707,-4.202,2.504,GB,0.616
15,NYG @ LA,10.525,8.111,-2.414,LA,0.716
7,LV @ LAC,8.368,6.138,-2.230,LAC,0.667
3,CLE @ TB,6.515,8.087,1.572,TB,0.715
10,NO @ BAL,6.394,7.936,1.542,BAL,0.712
13,SEA @ ARI,-7.002,-5.675,1.326,SEA,0.655
11,PHI @ TEN,-7.436,-8.581,-1.145,PHI,0.727


In [78]:
biggest_projection_changes["Preseason Margin"] = (
    biggest_projection_changes["Preseason Margin"].round(2)
)

biggest_projection_changes["Weekly Margin"] = (
    biggest_projection_changes["Weekly Margin"].round(2)
)

biggest_projection_changes["Margin Change"] = (
    biggest_projection_changes["Margin Change"].round(2)
)

biggest_projection_changes["Win Prob"] = (
    biggest_projection_changes["Win Prob"].round(3)
)

# Sanity Checks


In [79]:
assert len(games) == len(target_schedule)
assert games["weekly_neutral_strength_diff"].notna().all()
assert games["expected_home_margin"].notna().all()
assert games["predicted_win_probability"].between(0.5, 1.0).all()
assert games["projected_total"].notna().all()

score_margin_check = (
    games["projected_home_score"]
    - games["projected_away_score"]
)

score_total_check = (
    games["projected_home_score"]
    + games["projected_away_score"]
)

assert np.allclose(
    score_margin_check,
    games["expected_home_margin"]
)

assert np.allclose(
    score_total_check,
    games["projected_total"]
)

print("All weekly game prediction sanity checks passed.")


All weekly game prediction sanity checks passed.


# Save Week Outputs

Everything is written only to the weekly layer.

The detailed parquet preserves all modeling fields for later evaluation. The CSV files provide the clean weekly projection board and largest model versus market differences.


In [80]:
detailed_path = (
    WEEKLY_DATA_DIR
    / "week2_projections.parquet"
)

games.to_parquet(
    detailed_path,
    index=False
)

projection_save = projection_board.copy()
projection_save["Win Prob"] = projection_save["Win Prob"].round(3)
projection_save["ATS Diff"] = projection_save["ATS Diff"].round(2)
projection_save["Model Total"] = projection_save["Model Total"].round(1)
projection_save["Market Total"] = projection_save["Market Total"].round(1)
projection_save["Total Diff"] = projection_save["Total Diff"].round(1)
projection_save["Preseason Margin"] = projection_save["Preseason Margin"].round(2)
projection_save["Weekly Margin"] = projection_save["Weekly Margin"].round(2)
projection_save["Margin Change"] = projection_save["Margin Change"].round(2)

projection_save.to_csv(
    WEEK_OUTPUT_DIR
    / "week2_projections.csv",
    index=False
)

biggest_ats_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv",
    index=False
)

biggest_total_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv",
    index=False
)

biggest_projection_changes.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv",
    index=False
)

print("Saved:")
print(detailed_path)
print(WEEK_OUTPUT_DIR / "week2_projections.csv")
print(WEEK_OUTPUT_DIR / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv")
print(WEEK_OUTPUT_DIR / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv")
print(WEEK_OUTPUT_DIR / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv")


Saved:
..\..\data\processed\weekly\week2_projections.parquet
..\..\weekly_projections\outputs\week_02\week2_projections.csv
..\..\weekly_projections\outputs\week_02\week_02_biggest_ats_differences.csv
..\..\weekly_projections\outputs\week_02\week_02_biggest_total_differences.csv
..\..\weekly_projections\outputs\week_02\week_02_biggest_projection_changes.csv


# Weekly Pipeline

For each future week, the intended order is:

`02_Weekly_Data_Update`  
→ `03_Inseason_Feature_Engineering`  
→ `04_Weekly_Team_Strength`  
→ `06_Week2_Projections`

The original preseason notebooks remain frozen throughout the season.

The next notebook will be `06_Weekly_Evaluation.ipynb`, which will compare locked weekly predictions against actual results after each week is complete.
